# 1. Ручной градиентный бустинг и engineering признаков

На начало месяца предсказываем записанные покупки. 500 клиентов, 50 активов, top-5. Три внутренних временных блока используются для подбора, validation — для диагностики. Обучение выполнено CLI; этот notebook выполнен для просмотра и проверки результатов.

In [1]:
from pathlib import Path
import json, sys
import numpy as np
import pandas as pd
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "src/research/feature_automation").is_dir())
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
RUN_TRAINING = False
RESULT_DIR = ROOT / ("results/runs/notebook-reproduction" if RUN_TRAINING else "results/final")
DATA_DIR = ROOT / "data/processed"
read = lambda name: json.loads((RESULT_DIR / name).read_text())
print("Результаты фиксированного эксперимента; RUN_TRAINING =", RUN_TRAINING)


Результаты фиксированного эксперимента; RUN_TRAINING = False


In [2]:
if RUN_TRAINING:
    import subprocess
    subprocess.run([sys.executable, "-m", "research.feature_automation.experiment",
                    "--data-dir", str(DATA_DIR), "--out", str(RESULT_DIR), "--phase", "manual"],
                   cwd=ROOT, check=True)
else:
    print("Этап manual ранее выполнен CLI; ниже его журнал и результаты.")


Этап manual ранее выполнен CLI; ниже его журнал и результаты.


## Формулы и смысл

count — покупки; unique — разнообразие; recency — давность; share — доля интереса; trend — логарифмическое изменение соседних окон. Все события строго до cutoff. Календарные месяцы отличаются от фиксированных дней. Категории присоединяются по доступной версии.

In [3]:
from research.feature_automation.features import manual_features
specs = manual_features()
print(pd.DataFrame([s.description() for s in specs]).to_string(index=False))
print("Число ручных признаков:", len(specs))

   relation operation  days  months                       name
       user     count     0       1            user__count__1m
       pair     count     0       1            pair__count__1m
       item     count     0       1            item__count__1m
       user     count     0       3            user__count__3m
       pair     count     0       3            pair__count__3m
       user     count    30       0           user__count__30d
       item     count    30       0           item__count__30d
       pair     count    30       0           pair__count__30d
       user    unique    30       0          user__unique__30d
       item    unique    30       0          item__unique__30d
       pair     share    30       0           pair__share__30d
       user     count    90       0           user__count__90d
       item     count    90       0           item__count__90d
       pair     count    90       0           pair__count__90d
       user    unique    90       0          user__uniq

In [4]:
m = read("manual_stage.json")
for name in ("old_5", "manual"):
    print(name, "CV:", m["best"][name]["aggregate"], "params:", m["best"][name]["params"])
    print("validation:", m["frozen"][name]["validation"])
print("Правило validation:", m["rule_validation"])
print(pd.DataFrame(m["ablations"]).to_string(index=False))

old_5 CV: {'Recall@5': 0.5462320025488475, 'NDCG@5': 0.47094375520322745, 'Precision@5': 0.2066844919786096, 'LogLoss': 0.02759893906670934} params: {'learning_rate': 0.05, 'max_iter': 100, 'max_leaf_nodes': 7, 'min_samples_leaf': 20, 'l2_regularization': 0.0}
validation: {'Recall@5': 0.621705454397762, 'NDCG@5': 0.5307210697563147, 'Precision@5': 0.246985446985447, 'Precision@5_all': 0.0396, 'rows': 150000, 'positives': 1068, 'eligible_cases': 481, 'all_cases': 3000, 'LogLoss': 0.02750875472563528, 'ROC-AUC': 0.9292865828921647}
manual CV: {'Recall@5': 0.5996762855385849, 'NDCG@5': 0.5089004234633566, 'Precision@5': 0.22459893048128343, 'LogLoss': 0.025616977903537952} params: {'learning_rate': 0.05, 'max_iter': 100, 'max_leaf_nodes': 7, 'min_samples_leaf': 20, 'l2_regularization': 0.0}
validation: {'Recall@5': 0.6518641845564923, 'NDCG@5': 0.5601559540519894, 'Precision@5': 0.2573804573804574, 'Precision@5_all': 0.041266666666666674, 'rows': 150000, 'positives': 1068, 'eligible_cases

## Интерпретация

Разность при одинаковых параметрах отделяет влияние признаков от настройки. Исключение групп показывает условный вклад. Внутренние результаты участвовали в выборе, поэтому не являются независимым подтверждением.

In [5]:
print("Контроль одинаковых параметров:")
for name, r in m["controls"].items():
    print(name, r["aggregate"])
print("Профиль этапа:", m["timing"])

Контроль одинаковых параметров:
old_5 {'Recall@5': 0.5456218766379194, 'NDCG@5': 0.467351742440204, 'Precision@5': 0.20588235294117646, 'LogLoss': 0.027665550515034536}
manual {'Recall@5': 0.5902987201449769, 'NDCG@5': 0.5041883452703647, 'Precision@5': 0.2192513368983957, 'LogLoss': 0.02578198998629657}
Профиль этапа: {'fit_seconds': 511.6255392062012, 'predict_seconds': 13.441730915685184, 'feature_seconds': 13.142962626065128, 'fits': 113}
